In [10]:
import os
import random
import string
import math
from scipy import stats
import numpy as np

import hail as hl
from ukb_utils import hail_init
from ko_utils import ko
from ko_utils import io
from ko_utils import samples
from ukb_utils import genotypes

In [3]:
os.chdir('/well/lindgren/UKBIOBANK/flassen/projects/KO/wes_ko_ukbb')
hail_init.hail_bmrc_init('hail.log', 'GRCh38')

Running on Apache Spark version 3.1.2
SparkUI available at http://compa033.hpc.in.bmrc.ox.ac.uk:4042
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.77-684f32d73643
LOGGING: writing to hail.log


In [16]:
phenotypes = "data/phenotypes/filtered_phenotypes_cts.txt"
ht = hl.import_table(phenotypes,
             types={'eid': hl.tstr},
             missing=["",'""',"NA"],
             impute=True,
             force=True,
             key='eid'
             )

2022-06-09 11:27:39 Hail: INFO: Reading table to impute column types
2022-06-09 11:27:52 Hail: INFO: Loading 192 fields. Counts by type:
  float64: 185
  int32: 4
  bool: 2
  str: 1


In [17]:
values = ht.Alanine_aminotransferase.collect()

2022-06-09 11:27:56 Hail: INFO: Coerced sorted dataset


In [19]:
x = np.array(values)

In [21]:
x

array([2.53765721517353, None, 3.12763734443393, ..., 2.87299950817169,
       3.40386049908164, 2.69665215614984], dtype=object)

In [20]:
get_irnt(x)

TypeError: ufunc 'isnan' not supported for the input types, and the inputs could not be safely coerced to any supported types according to the casting rule ''safe''

In [14]:
#get_irnt(ht.Alanine_aminotransferase)

In [13]:
x_orig = ht.Alanine_aminotransferase_residual
is_def = hl.is_defined(x_orig)

In [22]:
def get_irnt(x, k = 3/8):
    """Inverse-rank normal transformation
    
    :param x: Continuous data to transform
    :param k: Adjustable offset" (default: Blom offset = 3/8). 
    Must be in interval (0, 1/2)
    """
    
    x_orig = np.asarray(x).copy()
    is_def = ~np.isnone(x_orig)
    x = x_orig[is_def]
    
    # Number of samples
    n = x.shape[0]
    
    # Rank of samples
    # rankdata returns 1-indexed rank
    # method='min' will give the minimum rank to tied values
    rank = stats.rankdata(x, method='min')
    
    # Transformation
    # Based on equation in this R function documentation:
    # https://cran.r-project.org/web/packages/RNOmni/vignettes/RNOmni.html#inverse-normal-transformation
    
    x_irnt = stats.norm.ppf((rank - k) / (n - 2*k + 1))
    x_orig[is_def] = x_irnt
    x_irnt = x_orig
    
    return x_irnt